# Contrastive training Mini-CLIP modela

U ovom delu implementiran je trening Mini-CLIP modela korišćenjem
**contrastive learning** pristupa.

Model se sastoji iz dva enkodera:
- image encoder-a, zasnovanog na ResNet18 arhitekturi;
- text encoder-a, zasnovanog na Transformer arhitekturi.

Oba enkodera mapiraju svoje ulaze u zajednički embedding prostor dimenzije 256.
Cilj treninga je da embedding odgovarajuće slike i njenog tekstualnog opisa
bude što sličniji, dok embedding-i nepovezanih image-caption parova treba
da budu međusobno manje slični.

Prvo se koristi ResNet18 implementiran od početka kao
image encoder i Transformer kao text encoder. Kasnije se isti postupak
ponavlja sa pretrained i fine-tuned ResNet18 varijantama.

In [1]:
import os
import json

import torch
import torch.nn.functional as F

from image_encoder import (
    ResNet18Encoder,
    ResNet18EncoderPretrained,
    ResNet18EncoderFineTuned
)

from text_encoder import MiniTextTransformer

In [2]:
%%capture
get_ipython().run_line_magic("run", "01_data_setup.ipynb")

### Učitavanje pripremljenih podataka

Dataset i DataLoader-i za contrastive training formirani su u
`01_data_setup.ipynb`. U ovoj svesci koriste se već pripremljeni
image-caption batch-evi, kako bi fokus ostao na generisanju embedding-a,
contrastive loss-u i treningu modela.

In [3]:
train_loader = contrastive_train_loader
val_loader = contrastive_val_loader

### Generisanje image i text embedding-a

Jedan batch se prosleđuje kroz oba encoder-a kako bi se dobile njihove
reprezentacije u zajedničkom embedding prostoru.

Slike iz batch-a prosleđuju se image encoder-u, dok se `input_ids` i
`attention_mask` vrednosti prosleđuju text encoder-u.

Pošto oba encoder-a vraćaju embedding vektore dimenzije 256, dobijene
reprezentacije mogu kasnije direktno da se porede pomoću kosinusne sličnosti.

In [4]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = "./data"
VOCAB_FILE = os.path.join(DATA_DIR, "processed", "vocab.json")

with open(VOCAB_FILE, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

token_to_id = vocab_data["token_to_id"]

VOCAB_SIZE = len(token_to_id)
MAX_LENGTH = vocab_data["max_length"]
PAD_ID = token_to_id["<PAD>"]

print("Vocabulary size:", VOCAB_SIZE)
print("Maximum caption length:", MAX_LENGTH)
print("PAD ID:", PAD_ID)

Vocabulary size: 7747
Maximum caption length: 32
PAD ID: 0


In [5]:
EMBEDDING_DIM = 256

image_encoder = ResNet18EncoderPretrained(
    emb_dim=EMBEDDING_DIM
).to(DEVICE)

text_encoder = MiniTextTransformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    padding_idx=PAD_ID,
    output_dim=EMBEDDING_DIM
).to(DEVICE)

In [6]:
batch = next(iter(train_loader))

images = batch["image"].to(DEVICE)
input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)

image_embeddings = image_encoder(images)

text_embeddings = text_encoder(input_ids, attention_mask)

print("Image embeddings:", image_embeddings.shape)
print("Text embeddings:", text_embeddings.shape)

Image embeddings: torch.Size([32, 256])
Text embeddings: torch.Size([32, 256])


### Contrastive loss

Contrastive loss se računa na osnovu sličnosti između image i text embedding-a.

Za svaki batch formira se matrica sličnosti između svih slika i svih caption-a.
Odgovarajući image-caption parovi nalaze se na dijagonali matrice - pozitivni parovi, dok ostali
parovi predstavljaju negativne primere.

Loss se računa simetrično u oba smera: svaka slika treba da prepozna svoj
caption, a svaki caption svoju odgovarajuću sliku.

Sličnosti se dodatno skaliraju parametrom `temperature`, koji kontroliše koliko
će razlike između sličnih i nesličnih parova biti izražene.

In [7]:
def clip_contrastive_loss(image_embeddings, text_embeddings, temperature=0.07):
    logits = (
        image_embeddings @ text_embeddings.T
    ) / temperature

    labels = torch.arange(logits.size(0), device=logits.device)

    image_to_text_loss = F.cross_entropy(logits, labels)

    text_to_image_loss = F.cross_entropy(logits.T, labels)

    loss = (image_to_text_loss + text_to_image_loss) / 2

    return loss

### Optimizer

Optimizer ažurira parametre image i text encoder-a na osnovu gradijenata
dobijenih iz contrastive loss-a.

Pošto se oba encoder-a treniraju zajedno, optimizer dobija parametre oba modela.
Za optimizaciju se koristi AdamW optimizer.

In [8]:
optimizer = torch.optim.AdamW(
    list(image_encoder.parameters()) +
    list(text_encoder.parameters()),
    lr=1e-4
)

### Training loop

Tokom jedne epohe model prolazi kroz sve batch-eve trening skupa.

Za svaki batch slike i caption-i se prosleđuju odgovarajućim encoder-ima,
nakon čega se dobijaju image i text embedding-i. Na osnovu njih računa se
contrastive loss.

Gradijenti dobijeni iz loss-a koriste se za ažuriranje parametara oba encoder-a
pomoću optimizer-a.

In [9]:
def train_one_epoch(
    image_encoder,
    text_encoder,
    train_loader,
    optimizer,
    device
):
    image_encoder.train()
    text_encoder.train()

    total_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        image_embeddings = image_encoder(images)

        text_embeddings = text_encoder(input_ids, attention_mask)

        loss = clip_contrastive_loss(image_embeddings,text_embeddings)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    return average_loss

### Validation loop

Tokom validacije model prolazi kroz validation skup kako bi se procenilo
koliko dobro radi na podacima koji nisu korišćeni za ažuriranje težina.

Za svaki batch računaju se image i text embedding-i i contrastive loss,
ali se ne računaju gradijenti i parametri modela se ne menjaju.

In [10]:
def validate_one_epoch(
    image_encoder,
    text_encoder,
    val_loader,
    device
):
    image_encoder.eval()
    text_encoder.eval()

    total_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            image_embeddings = image_encoder(images)

            text_embeddings = text_encoder(input_ids, attention_mask)

            loss = clip_contrastive_loss(
                image_embeddings,
                text_embeddings
            )

            total_loss += loss.item()

    average_loss = total_loss / len(val_loader)

    return average_loss

### Trening kroz više epoha i čuvanje najboljeg modela

Model se trenira kroz više epoha. U svakoj epohi najpre se prolazi kroz
training skup i ažuriraju se parametri modela, a zatim se na validation skupu
procenjuje koliko dobro trenutno istrenirani model generalizuje.

Training i validation loss se čuvaju nakon svake epohe kako bi kasnije mogli
da se analiziraju i prikažu grafički.

Nakon svake epohe proverava se validation loss. Ukoliko je trenutni
validation loss manji od najboljeg prethodno zabeleženog rezultata, čuva se
trenutno stanje image encoder-a, text encoder-a i optimizer-a.
Na ovaj način se nakon treninga zadržava model koji je ostvario najbolji
rezultat na validation skupu, a ne nužno model iz poslednje epohe.

In [11]:
MODEL_DIR = "./models"
os.makedirs(MODEL_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "pretrained_32bs_30e.pt"
)

best_val_loss = float("inf")

In [ ]:
NUM_EPOCHS = 30

train_losses = []
val_losses = []

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    train_loss = train_one_epoch(
        image_encoder,
        text_encoder,
        train_loader,
        optimizer,
        DEVICE
    )

    val_loss = validate_one_epoch(
        image_encoder,
        text_encoder,
        val_loader,
        DEVICE
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "image_encoder_state_dict":
                    image_encoder.state_dict(),
                "text_encoder_state_dict":
                    text_encoder.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "val_loss": val_loss
            },
            BEST_MODEL_PATH
        )

        print("Best model saved.")

TODO dodati vizuelizaciju train losses/val losses kroz epohe